# Training Analysis & Visualization

This notebook loads training metrics from WandB/SwanLab exports (or saved JSON logs),
compares Baseline vs. Joint training curves, and generates figures for the final report.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.metrics import compare_runs

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

In [ ]:
# Load training metrics (export from WandB/SwanLab or use saved JSON)
LOGS_DIR = Path("../outputs/logs")

baseline_path = LOGS_DIR / "act_baseline_metrics.json"
joint_path = LOGS_DIR / "act_joint_metrics.json"

if baseline_path.exists() and joint_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)
    with open(joint_path) as f:
        joint = json.load(f)
    print("Loaded real training metrics.")
else:
    # Generate synthetic data for demonstration
    print("No saved metrics found. Using synthetic data for demonstration.")
    rng = np.random.RandomState(42)
    epochs = 200
    
    baseline = {
        'train_loss': (np.exp(-np.arange(epochs) / 50) * 0.3 + 0.02 + rng.randn(epochs) * 0.005).tolist(),
        'val_loss': (np.exp(-np.arange(0, epochs, 5) / 45) * 0.28 + 0.025 + rng.randn(epochs // 5) * 0.003).tolist(),
    }
    joint = {
        'train_loss': (np.exp(-np.arange(epochs) / 45) * 0.25 + 0.018 + rng.randn(epochs) * 0.004).tolist(),
        'val_loss': (np.exp(-np.arange(0, epochs, 5) / 40) * 0.22 + 0.015 + rng.randn(epochs // 5) * 0.003).tolist(),
    }

In [ ]:
# Training loss curves (smoothed)
def smooth(y, w=10):
    return np.convolve(y, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
ax = axes[0]
for label, data, c in [('Baseline (Env A)', baseline['train_loss'], '#2196F3'),
                         ('Joint (Env A+B+C)', joint['train_loss'], '#FF5722')]:
    raw = np.array(data)
    ax.plot(smooth(raw, 15), color=c, label=label, linewidth=1.5)
    ax.plot(raw, color=c, alpha=0.08, linewidth=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Action L1 Loss')
ax.set_title('Training Loss (Smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[1]
for label, data, c in [('Baseline (Env A)', baseline['val_loss'], '#2196F3'),
                         ('Joint (Env A+B+C)', joint['val_loss'], '#FF5722')]:
    ax.plot(data, color=c, label=label, linewidth=1.5, marker='o', markersize=2)
ax.set_xlabel('Validation Step')
ax.set_ylabel('Action L1 Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quantitative comparison
result = compare_runs(
    {'train_loss': baseline['train_loss'], 'val_loss': baseline['val_loss']},
    {'train_loss': joint['train_loss'], 'val_loss': joint['val_loss']},
)

print("=" * 50)
print("BASELINE vs JOINT — Training Comparison")
print("=" * 50)
print(f"{'Metric':<30} {'Baseline':>12} {'Joint':>12}")
print("=" * 50)
print(f"{'Best Validation Loss':<30} {result['baseline']['best_val_loss']:>12.6f} {result['joint']['best_val_loss']:>12.6f}")
print(f"{'Final Training Loss':<30} {result['baseline']['final_train_loss']:>12.6f} {result['joint']['final_train_loss']:>12.6f}")
print(f"{'Convergence Epoch':<30} {result['baseline']['convergence_epoch']:>12} {result['joint']['convergence_epoch']:>12}")
print("=" * 50)
print(f"Verdict: {result['verdict']}")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Best Val Loss', 'Final Train Loss']
baseline_vals = [result['baseline']['best_val_loss'], result['baseline']['final_train_loss']]
joint_vals = [result['joint']['best_val_loss'], result['joint']['final_train_loss']]

x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, baseline_vals, width, label='Baseline (Env A)', color='#2196F3')
ax.bar(x + width/2, joint_vals, width, label='Joint (Env A+B+C)', color='#FF5722')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Action L1 Loss')
ax.set_title('Training Performance Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../outputs/figures/loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()